In [ ]:
!pip -q install scikit-learn openpyxl pandas numpy joblib

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ============================================================
# 1) General Settings
# ============================================================
TRAIN_PATH = "train.xlsx"
VALID_PATH = "valid.xlsx"
TEST_PATH  = "test.xlsx"

TEXT_COL = "clean_text"
LABEL_DIALECT_COL = "dialect"
LABEL_HATE_COL    = "hate"
LABEL_TOPIC_COL   = "topic"
LABELS = [LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL]

TFIDF_MAX_FEATURES = 50000
NGRAM_RANGE = (1, 2)
MIN_DF = 2
NB_ALPHA = 1.0
OUTPUT_DIR = "./naive_bayes_multilabel_with_emoji_feature"

id2dialect = {1: "Saudi", 0: "Egyptian"}
id2hate    = {1: "Hate", 0: "Not Hate"}
id2topic   = {1: "Religious", 0: "Political"}

HATE_EMOJIS = set([
    '💦', '🐖', '🐷', '🐽', '👞', '🐕', '🐶', '💩', '🐄', '🐮',
    '🐑', '🐏', '👎', '😡', '🤬', '👺', '👿', '😠'
])

# ============================================================
# 2) Emoji Feature
# ============================================================
def emoji_flag(text: str) -> int:
    text = "" if pd.isna(text) else str(text)
    return 1 if any(e in text for e in HATE_EMOJIS) else 0

class EmojiFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.Series):
            features = X.apply(lambda text: emoji_flag(text))
        else:
            features = pd.Series(X).apply(lambda text: emoji_flag(text))
        return features.values.reshape(-1, 1)

# ============================================================
# 3) Load + Validate Data - Clean Text Already Preprocessed
# ============================================================
def load_excel(path: str) -> pd.DataFrame:
    return pd.read_excel(path, engine="openpyxl")

def load_and_validate_xlsx(path: str) -> pd.DataFrame:
    df = load_excel(path)

    required = {TEXT_COL, LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {missing}\nAvailable columns: {list(df.columns)}")

    df = df.copy()
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()

    for c in LABELS:
        if df[c].isna().any():
            bad_rows = df[df[c].isna()].head(5)
            raise ValueError(f"{path}: column '{c}' has NaN values. Examples:\n{bad_rows[[TEXT_COL, c]]}")

        df[c] = pd.to_numeric(df[c], errors="raise").astype(int)
        bad = ~df[c].isin([0, 1])
        if bad.any():
            examples = df.loc[bad, [TEXT_COL, c]].head(5)
            raise ValueError(f"{path}: column '{c}' must be 0/1 only. Examples:\n{examples}")

    return df

print("📂 جاري تجهيز البيانات...")
train_df = load_and_validate_xlsx(TRAIN_PATH)
valid_df = load_and_validate_xlsx(VALID_PATH)
test_df  = load_and_validate_xlsx(TEST_PATH)

print("Rows:", len(train_df), len(valid_df), len(test_df))
print("Train label counts:")
print("dialect:", train_df[LABEL_DIALECT_COL].value_counts().to_dict())
print("hate:",    train_df[LABEL_HATE_COL].value_counts().to_dict())
print("topic:",   train_df[LABEL_TOPIC_COL].value_counts().to_dict())

# ============================================================
# 4) Prepare Inputs
# ============================================================
X_train = train_df[[TEXT_COL]]
X_valid = valid_df[[TEXT_COL]]
X_test  = test_df[[TEXT_COL]]

y_train = train_df[LABELS].values.astype(int)
y_valid = valid_df[LABELS].values.astype(int)
y_test  = test_df[LABELS].values.astype(int)

# ============================================================
# 5) Build Model
# ============================================================
features_pipeline = ColumnTransformer([
    ("tfidf", TfidfVectorizer(
        ngram_range=NGRAM_RANGE,
        min_df=MIN_DF,
        max_features=TFIDF_MAX_FEATURES,
        token_pattern=r"[^\s]+"
    ), TEXT_COL),
    ("emoji", EmojiFeatureExtractor(), TEXT_COL)
])

model = Pipeline([
    ("features", features_pipeline),
    ("clf", OneVsRestClassifier(MultinomialNB(alpha=NB_ALPHA)))
])

# ============================================================
# 6) Train
# ============================================================
print("\n⏳ جاري التدريب (Naive Bayes Multilabel)...")
model.fit(X_train, y_train)

# ============================================================
# 7) Predict on Validation + Test
# ============================================================
y_valid_pred = model.predict(X_valid)
y_test_pred  = model.predict(X_test)

# ============================================================
# 8) Metrics
# ============================================================
def compute_metrics(y_true, y_pred):
    metrics = {}
    metrics["overall_acc"] = np.mean(np.all(y_true == y_pred, axis=1))

    metrics["dialect_acc"] = accuracy_score(y_true[:, 0], y_pred[:, 0])
    metrics["dialect_f1"] = f1_score(y_true[:, 0], y_pred[:, 0], average="macro", zero_division=0)
    metrics["dialect_f1_micro"] = f1_score(y_true[:, 0], y_pred[:, 0], average="micro", zero_division=0)

    metrics["hate_acc"] = accuracy_score(y_true[:, 1], y_pred[:, 1])
    metrics["hate_f1"] = f1_score(y_true[:, 1], y_pred[:, 1], average="macro", zero_division=0)
    metrics["hate_f1_micro"] = f1_score(y_true[:, 1], y_pred[:, 1], average="micro", zero_division=0)

    metrics["topic_acc"] = accuracy_score(y_true[:, 2], y_pred[:, 2])
    metrics["topic_f1"] = f1_score(y_true[:, 2], y_pred[:, 2], average="macro", zero_division=0)
    metrics["topic_f1_micro"] = f1_score(y_true[:, 2], y_pred[:, 2], average="micro", zero_division=0)

    metrics["avg_f1"] = (metrics["dialect_f1"] + metrics["hate_f1"] + metrics["topic_f1"]) / 3.0
    metrics["avg_f1_micro"] = (
        metrics["dialect_f1_micro"] + metrics["hate_f1_micro"] + metrics["topic_f1_micro"]
    ) / 3.0

    return metrics

valid_metrics = compute_metrics(y_valid, y_valid_pred)
test_metrics  = compute_metrics(y_test, y_test_pred)

print("\nValidation Metrics:")
for k, v in valid_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nTest Metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

print(f"\nOverall exact-match accuracy (all labels together): {np.mean(np.all(y_test == y_test_pred, axis=1)):.4f}")

# ============================================================
# 9) Detailed Reports
# ============================================================
def print_report_with_micro(y_true, y_pred, title: str):
    rep = classification_report(y_true, y_pred, digits=4, zero_division=0, output_dict=True)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    def row(k):
        return rep.get(k, {"precision": 0.0, "recall": 0.0, "f1-score": 0.0, "support": 0})

    r0 = row("0")
    r1 = row("1")
    rmacro = row("macro avg")
    rweight = row("weighted avg")
    total_support = int(r0["support"]) + int(r1["support"])


    print(f"\n--- {title} ---")
    print(f"{'':>14}{'precision':>10}{'recall':>10}{'f1-score':>10}{'f1-micro':>10}{'support':>10}")
    print(f"{'0.0':>14}{r0['precision']:>10.4f}{r0['recall']:>10.4f}{r0['f1-score']:>10.4f}{f1_micro:>10.4f}{int(r0['support']):>10}")
    print(f"{'1.0':>14}{r1['precision']:>10.4f}{r1['recall']:>10.4f}{r1['f1-score']:>10.4f}{f1_micro:>10.4f}{int(r1['support']):>10}")
    print()
    print(f"{'accuracy':>14}{'':>10}{'':>10}{acc:>10.4f}{f1_micro:>10.4f}{total_support:>10}")
    print(f"{'macro avg':>14}{rmacro['precision']:>10.4f}{rmacro['recall']:>10.4f}{rmacro['f1-score']:>10.4f}{f1_micro:>10.4f}{int(rmacro['support']):>10}")
    print(f"{'weighted avg':>14}{rweight['precision']:>10.4f}{rweight['recall']:>10.4f}{rweight['f1-score']:>10.4f}{f1_micro:>10.4f}{int(rweight['support']):>10}")

print_report_with_micro(y_test[:, 0], y_test_pred[:, 0], "Dialect report (1=saudi,0=egyptian)")
print_report_with_micro(y_test[:, 1], y_test_pred[:, 1], "Hate report (1=hate,0=not hate)")
print_report_with_micro(y_test[:, 2], y_test_pred[:, 2], "Topic report (1=religious,0=political)")

# ============================================================
# 10) Save Model
# ============================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(model, os.path.join(OUTPUT_DIR, "naive_bayes_multilabel_model.joblib"))
print("\nSaved to:", OUTPUT_DIR)

# ============================================================
# 11) Predict Function
# ============================================================
def predict(text: str, threshold: float = 0.5):
    cleaned = "" if pd.isna(text) else str(text).strip()
    df_input = pd.DataFrame({TEXT_COL: [cleaned]})

    probs_c1 = model.predict_proba(df_input)[0]
    preds = (probs_c1 >= threshold).astype(int)

    dialect_id = int(preds[0])
    hate_id = int(preds[1])
    topic_id = int(preds[2])

    dialect_p = float(probs_c1[0])
    hate_p = float(probs_c1[1])
    topic_p = float(probs_c1[2])

    return {
        "dialect_id": dialect_id,
        "hate_id": hate_id,
        "topic_id": topic_id,
        "dialect_label": id2dialect[dialect_id],
        "hate_label": id2hate[hate_id],
        "topic_label": id2topic[topic_id],
        "dialect_probs": {"Egyptian": 1 - dialect_p, "Saudi": dialect_p},
        "hate_probs": {"Not Hate": 1 - hate_p, "Hate": hate_p},
        "topic_probs": {"Political": 1 - topic_p, "Religious": topic_p},
        "dialect_confidence": dialect_p if dialect_id == 1 else (1 - dialect_p),
        "hate_confidence": hate_p if hate_id == 1 else (1 - hate_p),
        "topic_confidence": topic_p if topic_id == 1 else (1 - topic_p),
        "emoji_flag": int(emoji_flag(text)),
    }

# ============================================================
# 12) Interactive Mode
# ============================================================
print("\n=== Interactive Mode ===")
print("Type: exit / quit / stop to end\n")

while True:
    text = input("Tweet> ").strip()
    if text.lower() in {"exit", "quit", "stop"}:
        print("Done.")
        break
    if not text:
        continue

    result = predict(text, threshold=0.5)

    print(
        f"Dialect: {result['dialect_label']} "
        f"(confidence={result['dialect_confidence']:.3f}, Egyptian={result['dialect_probs']['Egyptian']:.3f}, Saudi={result['dialect_probs']['Saudi']:.3f})"
    )
    print(
        f"Hate: {result['hate_label']} "
        f"(confidence={result['hate_confidence']:.3f}, Not Hate={result['hate_probs']['Not Hate']:.3f}, Hate={result['hate_probs']['Hate']:.3f}, emoji_flag={result['emoji_flag']})"
    )
    print(
        f"Topic: {result['topic_label']} "
        f"(confidence={result['topic_confidence']:.3f}, Political={result['topic_probs']['Political']:.3f}, Religious={result['topic_probs']['Religious']:.3f})"
    )

📂 جاري تجهيز البيانات...
Rows: 2773 595 594
Train label counts:
dialect: {1: 1417, 0: 1356}
hate: {1: 1443, 0: 1330}
topic: {0: 1407, 1: 1366}

⏳ جاري التدريب (Naive Bayes Multilabel)...

Validation Metrics:
overall_acc: 0.6538
dialect_acc: 0.8874
dialect_f1: 0.8873
dialect_f1_micro: 0.8874
hate_acc: 0.7580
hate_f1: 0.7573
hate_f1_micro: 0.7580
topic_acc: 0.9630
topic_f1: 0.9630
topic_f1_micro: 0.9630
avg_f1: 0.8692
avg_f1_micro: 0.8695

Test Metrics:
overall_acc: 0.6465
dialect_acc: 0.8822
dialect_f1: 0.8821
dialect_f1_micro: 0.8822
hate_acc: 0.7458
hate_f1: 0.7455
hate_f1_micro: 0.7458
topic_acc: 0.9545
topic_f1: 0.9545
topic_f1_micro: 0.9545
avg_f1: 0.8607
avg_f1_micro: 0.8608

Overall exact-match accuracy (all labels together): 0.6465

--- Dialect report (1=saudi,0=egyptian) ---
               precision    recall  f1-score  f1-micro   support
           0.0    0.9336    0.8396    0.8841    0.8822       318
           1.0    0.8344    0.9312    0.8801    0.8822       276

      accu